In [1]:
from envs.mh5robotenv import MH5RobotEnv

from tensordict.nn import  TensorDictModule
from tensordict.nn.distributions import NormalParamExtractor

import torch
from torch import nn

from torchrl.envs.utils import check_env_specs
from torchrl.envs import (
    GymEnv,
    TransformedEnv,
    Compose,
    ObservationNorm,
    DoubleToFloat,
    StepCounter,
)
from torchrl.modules import ProbabilisticActor, TanhNormal, ValueOperator


In [3]:
device = torch.device("cpu")
if torch.backends.mps.is_available():
    device = torch.device("mps")
if torch.cuda.is_available():
    device = torch.device("cuda")
print(device)

mps


In [4]:
base_env = GymEnv("MH5Robot-v8", device=device)

In [5]:
env = TransformedEnv(
    base_env,
    Compose(
        ObservationNorm(in_keys=['observation']),
        DoubleToFloat(),
        StepCounter(),
    ),
)

In [6]:
env.transform[0].init_stats(num_iter=1000, reduce_dim=0, cat_dim=0)
print("normalization constant shape:", env.transform[0].loc.shape)

normalization constant shape: torch.Size([633])


In [7]:
print("observation_spec:", env.observation_spec)
print("reward_spec:", env.reward_spec)
print("input_spec:", env.input_spec)
print("action_spec (as defined by input_spec):", env.action_spec)

observation_spec: Composite(
    observation: UnboundedContinuous(
        shape=torch.Size([633]),
        space=ContinuousBox(
            low=Tensor(shape=torch.Size([633]), device=mps:0, dtype=torch.float32, contiguous=True),
            high=Tensor(shape=torch.Size([633]), device=mps:0, dtype=torch.float32, contiguous=True)),
        device=mps:0,
        dtype=torch.float32,
        domain=continuous),
    step_count: BoundedDiscrete(
        shape=torch.Size([1]),
        space=ContinuousBox(
            low=Tensor(shape=torch.Size([1]), device=mps:0, dtype=torch.int64, contiguous=True),
            high=Tensor(shape=torch.Size([1]), device=mps:0, dtype=torch.int64, contiguous=True)),
        device=mps:0,
        dtype=torch.int64,
        domain=discrete),
    device=mps:0,
    shape=torch.Size([]),
    data_cls=None)
reward_spec: UnboundedContinuous(
    shape=torch.Size([1]),
    space=ContinuousBox(
        low=Tensor(shape=torch.Size([1]), device=mps:0, dtype=torch.float32

In [8]:
check_env_specs(env)

2025-10-13 18:39:04,711 [torchrl][INFO]    check_env_specs succeeded! [END]


In [9]:
rollout = env.rollout(3)
print("rollout of three steps:", rollout)
print("Shape of the rollout TensorDict:", rollout.batch_size)

rollout of three steps: TensorDict(
    fields={
        action: Tensor(shape=torch.Size([3, 24]), device=mps:0, dtype=torch.float32, is_shared=False),
        done: Tensor(shape=torch.Size([3, 1]), device=mps:0, dtype=torch.bool, is_shared=False),
        next: TensorDict(
            fields={
                done: Tensor(shape=torch.Size([3, 1]), device=mps:0, dtype=torch.bool, is_shared=False),
                observation: Tensor(shape=torch.Size([3, 633]), device=mps:0, dtype=torch.float32, is_shared=False),
                reward: Tensor(shape=torch.Size([3, 1]), device=mps:0, dtype=torch.float32, is_shared=False),
                step_count: Tensor(shape=torch.Size([3, 1]), device=mps:0, dtype=torch.int64, is_shared=False),
                terminated: Tensor(shape=torch.Size([3, 1]), device=mps:0, dtype=torch.bool, is_shared=False),
                truncated: Tensor(shape=torch.Size([3, 1]), device=mps:0, dtype=torch.bool, is_shared=False)},
            batch_size=torch.Size([3])

In [10]:
config = {
    'num_cells': 256,
}

In [11]:
actor_net = nn.Sequential(
    nn.LazyLinear(config['num_cells'], device=device),
    nn.Tanh(),
    nn.LazyLinear(config['num_cells'], device=device),
    nn.Tanh(),
    nn.LazyLinear(config['num_cells'], device=device),
    nn.Tanh(),
    nn.LazyLinear(2 * env.action_spec.shape[-1], device=device),
    NormalParamExtractor(),
)

policy_module = TensorDictModule(actor_net, in_keys=['observation'], out_keys=['loc', 'scale'])

policy_module = ProbabilisticActor(
    module=policy_module,
    spec=env.action_spec,
    in_keys=['loc', 'scale'],
    distribution_class=TanhNormal,
    distribution_kwargs={
        'low': env.action_spec_unbatched.space.low,
        'high': env.action_spec_unbatched.space.high,
    },
    return_log_prob=True,
)

In [12]:
value_net = nn.Sequential(
    nn.LazyLinear(config['num_cells'], device=device),
    nn.Tanh(),
    nn.LazyLinear(config['num_cells'], device=device),
    nn.Tanh(),
    nn.LazyLinear(config['num_cells'], device=device),
    nn.Tanh(),
    nn.LazyLinear(1, device=device)
)

value_module = ValueOperator(
    module=value_net,
    in_keys=['observation'],
)

In [13]:
print("Running policy:", policy_module(env.reset()))
print("Running value:", value_module(env.reset()))

Running policy: TensorDict(
    fields={
        action: Tensor(shape=torch.Size([24]), device=mps:0, dtype=torch.float32, is_shared=False),
        action_log_prob: Tensor(shape=torch.Size([]), device=mps:0, dtype=torch.float32, is_shared=False),
        done: Tensor(shape=torch.Size([1]), device=mps:0, dtype=torch.bool, is_shared=False),
        loc: Tensor(shape=torch.Size([24]), device=mps:0, dtype=torch.float32, is_shared=False),
        observation: Tensor(shape=torch.Size([633]), device=mps:0, dtype=torch.float32, is_shared=False),
        scale: Tensor(shape=torch.Size([24]), device=mps:0, dtype=torch.float32, is_shared=False),
        step_count: Tensor(shape=torch.Size([1]), device=mps:0, dtype=torch.int64, is_shared=False),
        terminated: Tensor(shape=torch.Size([1]), device=mps:0, dtype=torch.bool, is_shared=False),
        truncated: Tensor(shape=torch.Size([1]), device=mps:0, dtype=torch.bool, is_shared=False)},
    batch_size=torch.Size([]),
    device=mps:0,
    is

In [16]:
env.action_spec_unbatched.space.low

tensor([-0.3491, -1.5708, -1.5708,  0.0000, -1.5708, -1.8850,  0.0000, -1.5708,
         0.0000, -1.5708, -1.8850,  0.0000, -0.7854, -1.8850,  0.0000, -0.7854,
        -1.5708, -1.5708, -0.7854, -1.8850,  0.0000, -0.7854, -1.5708, -1.5708],
       device='mps:0')

In [15]:
dir(base_env._env.env.env.model)

['__class__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '_address',
 '_from_model_ptr',
 '_pybind11_conduit_v1_',
 '_size_fields',
 '_sizes',
 'actuator',
 'actuator_acc0',
 'actuator_actadr',
 'actuator_actearly',
 'actuator_actlimited',
 'actuator_actnum',
 'actuator_actrange',
 'actuator_biasprm',
 'actuator_biastype',
 'actuator_cranklength',
 'actuator_ctrllimited',
 'actuator_ctrlrange',
 'actuator_dynprm',
 'actuator_dyntype',
 'actuator_forcelimited',
 'actuator_forcerange',
 'actuator_gainprm',
 'actuator_gaintype',
 'actuator_gear',
 'actuator_group',
 'actuator_length0',
 'actuator_lengthrange',
 'actuator_plugin',
 'actuator_trnid

In [ ]:
base_env._env.env.env.model.actuator('head_p')